In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path('/Users/milenakulik/PycharmProjects/ml-wakacyjne-wyzwanie-2026/01_przetwarzanie_wizualizacja_danych')

In [26]:
titanic_df = pd.read_csv(base_path / 'titanic.csv', index_col='PassengerId')

#zastąpienie kolumn SibSp i Parch kolumną FamilyMemers
titanic_df['FamilyMembers']=titanic_df['SibSp']+titanic_df['Parch']
titanic_df=titanic_df.drop(columns=['Ticket', 'SibSp', 'Parch'])

#zastąpienie kolumny Name kolumną Title
titanic_df['Name']=titanic_df['Name'].str.extract(r',\s([^\.]+)\.')
titanic_df=titanic_df.rename(columns={'Name':'Title'})

#zastąpienie kolumny Cabin kolumną Has_Cabin
titanic_df['Cabin']=titanic_df['Cabin'].notna().astype(int)
titanic_df=titanic_df.rename(columns={'Cabin':'Has_Cabin'})

#zmiana typów danych
titanic_df['Title']=titanic_df['Title'].astype(object)
titanic_df['Pclass']=titanic_df['Pclass'].astype(object)
titanic_df['Sex']=titanic_df['Sex'].astype(object)
titanic_df['Embarked']=titanic_df['Embarked'].astype(object)

#uzupełnienie brakujących wartości w kolumnie Embarked
titanic_df['Embarked']=titanic_df['Embarked'].fillna(titanic_df['Embarked'].mode()[0])

#uzupełnienie brakujących wartości w kolumnie Age
titanic_df['Age']=titanic_df['Age'].fillna(titanic_df.groupby('Title').Age.transform('median'))

#uzupełnienie brakujących wartości w kolumnie Fare
titanic_df['Fare']=titanic_df['Fare'].fillna(titanic_df.groupby(['Pclass','Embarked']).Fare.transform('median'))



# Brakujące dane

In [27]:
titanic_df.isna().sum()

Survived         0
Pclass           0
Title            0
Sex              0
Age              0
Fare             0
Has_Cabin        0
Embarked         0
FamilyMembers    0
dtype: int64

In [28]:
#uzupełnienie kolumny Age - zgodnie z rozkładem istniejących danych
def get_column_distribution(df, column):
    return df[column].value_counts(normalize=True)

age_count_prob=get_column_distribution(titanic_df, 'Age')
age_count_prob

age_values=age_count_prob.index
age_probs=age_count_prob.values

missing_age_vals=titanic_df.Age.isna()
titanic_df.loc[missing_age_vals, 'Age']=np.random.choice(age_values, size=missing_age_vals.sum(), p=age_probs)

In [29]:
#uzupełnienie kolumny Fare - zgodnie z rozkładem istniejących danych
fare_count_prob=get_column_distribution(titanic_df, 'Fare')
fare_count_prob

fare_values=fare_count_prob.index
fare_probs=fare_count_prob.values

missing_fare_vals=titanic_df.Fare.isna()
titanic_df.loc[missing_fare_vals, 'Fare']=np.random.choice(fare_values, size=missing_fare_vals.sum(), p=fare_probs)

In [30]:
#uzupełnienie kolumny Embarked - najczęstsza wartość (moda)
titanic_df['Embarked']=titanic_df['Embarked'].fillna(titanic_df['Embarked'].mode()[0])

zamiana kolumny Cabin na Deck i uzupełnienie danych w następnej sekcji

# Dodanie (zamienienie) kolumn
- pełne imię i nazwisko nie będzie użyteczne, ale wyodrębnię tytuł (dużo unikalnych wartości)
- pełny numer biletu również ma dużo unikalnych wartości, wyodrębnię więc początek (sekcję)
- tak samo robię z numerem kabiny, zostawiam tylko pierwszą literkę (tam, gdzie brakowało Cabin, Deck to Unknown)

In [31]:
#zamienienie kolumny Name na Title poprzez wyodrębnienie tytułu z kolumny Name
titanic_df['Name']=titanic_df['Name'].str.extract(r',\s([^\.]+)\.')
titanic_df=titanic_df.rename(columns={'Name':'Title'})

KeyError: 'Name'

In [ ]:
#dodanie kolumny FamilySize (wielkość rodziny)
titanic_df['FamilySize']=titanic_df['SibSp']+titanic_df['Parch']+1

In [ ]:
#zamienienie kolumny Ticket na TicketType poprzez wyodrębnienie "prefiksu" numeru biletu
titanic_df['Ticket']=titanic_df['Ticket'].str.split(' ').str[0]
titanic_df.loc[titanic_df['Ticket'].str.isdigit(), 'Ticket']='None'
titanic_df=titanic_df.rename(columns={'Ticket':'TicketType'})

In [ ]:
#zamienienie kolumny Cabin na Deck poprzez zredukowanie wartości z kolumny Cabin do pierwszej litery
titanic_df['Cabin']=titanic_df['Cabin'].str[0]
titanic_df=titanic_df.rename(columns={'Cabin':'Deck'})

#uzupełnianie kolumny Deck - "Unknown"
titanic_df['Deck']=titanic_df['Deck'].fillna('Unknown')

# Outliery

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#dla danych numerycznych
for column in ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=titanic_df[column])
    plt.title(column)
    plt.show()

#### dla Age
pojedyncze skrajne wartości nie muszą być błędami
#### dla Fare
wyraźne wartości skrajne, ale mogą być faktycznymi cenami
#### dla SibSp, Parch i FamilySize
są wartości skrajne, jednak są realne

In [ ]:
#dla danych kategorialnych
for column in ['Title', 'Sex', 'Embarked', 'Deck', 'TicketType']:
    print('\n', column)
    print(titanic_df[column].value_counts())

Występują rzadkie wartości w Title i TicketType, zostaną one połączone przeze mnie w kategorię Rare.

# Rozkład zmiennej predykowanej względem zmiennych kategorialnych i numerycznych

In [ ]:
#łączę rzadko występujące tytuły w kategorię Rare
title_counts=titanic_df['Title'].value_counts()
rare_titles=title_counts[title_counts<10].index
titanic_df['Title']=titanic_df['Title'].replace(rare_titles, 'Rare')

In [ ]:
#łączę rzadko występujące typy biletów w kategorię Rare
ticket_counts=titanic_df['TicketType'].value_counts()
rare_tickets=ticket_counts[ticket_counts<10].index
titanic_df['TicketType']=titanic_df['TicketType'].replace(rare_tickets, 'Rare')

In [ ]:
#survived względem zmiennych kategorialnych
categorical=['Sex', 'Title', 'Deck', 'Pclass', 'Embarked', 'TicketType']

for column in categorical:
    plt.figure(figsize=(6,4))
    sns.countplot(data=titanic_df, x=column, hue='Survived')
    plt.title(f'Survived vs {column}')
    plt.xlabel(column)
    plt.ylabel('Number of passengers')
    plt.show()

#### Sex vs Survived
wyraźna zależność - dla kobiet odsetek osób, które przeżyły jest znacznie większy niż odsetek ofiar, a dla mężczyzn jest odwrotnie
płeć może mieć znaczący wpływ na przewidywanie przeżycia pasażera

#### Title vs Survived
potwierdzają się wnioski z poprzedniego punktu, znacznie większa przeżywalność dla osób o tytułach żeńskich

#### Deck vs Survived
osoby, których numer kabiny nie jest znany, przeżywały rzadziej niż osoby, których numer kabiny jest znany

#### Pclass vs Survived
największy odsetek osób, które przeżyły, jest w klasie 1, najmniejszy w klasie 3

#### Embarked vs Survived
największy odsetek osób, które przeżyły, wsiadło w C, najmniejszy w S

#### TicketType vs Survived
największy odsetek osób, które przeżyły, miało bilet o numerze rozpoczynającym się od PC

In [ ]:
#survived względem zmiennych numerycznych
numeric=['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']

for column in numeric:
    plt.figure(figsize=(6,8))
    sns.boxplot(data=titanic_df, x='Survived', y=column)

    plt.title(f'{column} by Survival')
    plt.xlabel('Survived')
    plt.ylabel(column)
    plt.show()

#### dla Age
brak wyraźnej zależności, można pokusić się o stwierdzenie, że młodsze osoby przeżywały częściej

#### dla Fare
mediana ceny biletu dla osób, które przeżyły, jest większa niż mediana ceny biletu dla ofiar, więc im wyższa cena, tym większa przeżywalność (prawdopodobnie związane z klasami, w 1 przeżywalność była największa)

#### dla SibSp
brak zależności

#### dla Parch
posiadanie większej liczby rodziców/dzieci na pokładzie było częstsze u osób, które przeżyły

#### dla FamilySize
wśród ocalałych większe zróżnicowanie wielkości rodzin, osoby z większą rodziną przeżywały częściej

# Kodowanie kolumn

In [ ]:
titanic_df.dtypes
columns_to_code=titanic_df.select_dtypes(include='object').columns.tolist()
columns_to_code

In [ ]:
#kodowanie kolumny Pclass
title_counts=titanic_df['Pclass'].value_counts()

titanic_df=pd.get_dummies(data=titanic_df, prefix='Pclass', columns=['Pclass'], dtype=int)

In [ ]:
#kodowanie kolumny Title
title_counts=titanic_df['Title'].value_counts()

titanic_df=pd.get_dummies(data=titanic_df, prefix='Title', columns=['Title'], dtype=int)

In [ ]:
#kodowanie kolumny Sex
sex_counts=titanic_df['Sex'].value_counts()
titanic_df=pd.get_dummies(data=titanic_df, prefix='Sex', columns=['Sex'], dtype=int)

In [ ]:
#kodowanie kolumny TicketType
ticket_counts=titanic_df['TicketType'].value_counts()

titanic_df=pd.get_dummies(data=titanic_df, prefix='TicketType', columns=['TicketType'], dtype=int)

In [ ]:
#kodowanie kolumny Deck
deck_counts=titanic_df['Deck'].value_counts()

titanic_df=pd.get_dummies(data=titanic_df, prefix='Deck', columns=['Deck'], dtype=int)

In [ ]:
#kodowanie kolumny Embarked
embarked_counts = titanic_df['Embarked'].value_counts()

titanic_df = pd.get_dummies(data=titanic_df, prefix='Embarked', columns=['Embarked'], dtype=int)

# Wizualizacja danych

In [ ]:
#dla danych numerycznych
numerical=['Survived', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']

corr_matrix=titanic_df[numerical].corr()

plt.figure(figsize=(6,4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Data')
plt.tight_layout()
plt.show()

#### Fare
najbardziej dodatnio skorelowane z Survived, potwierdzają się wcześniejsze wnioski
kluczowa zmienna numeryczna do przewidywania przeżycia

#### Parch
lekko dodatnia korelacja Parch z Survived, potwierdzają się wcześniejsze wnioski

#### Age
lekko ujemna korelacja Age z Survived, potwierdzają się poprzednie wnioski
ujemne korelacje z Parch, SibSp, FamilySize (młodzi częściej podróżowali z rodziną, starsi częściej w pojedynkę)



In [ ]:
sns.countplot(data=titanic_df, x='Survived')
plt.title('Distribution of Survival')
plt.xlabel('Survived')
plt.ylabel('Number of passengers')
plt.show()

# Dystrybucja danych

In [ ]:
def plot_numeric_histogram(df, column_name):

    data=df[column_name]
    mean_val=data.mean()
    median_val=data.median()

    plt.figure(figsize=(6,4))
    plt.hist(data, bins=30, color='steelblue', edgecolor='black')
    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean:{mean_val:.2f}')
    plt.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median:{median_val:.2f}')

    plt.title(f'Distribution of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_numeric_histogram(titanic_df, 'Age') #rozkład asymetryczny prawostronnie (lekko)
plot_numeric_histogram(titanic_df, 'SibSp') #rozkład asymetryczny prawostronnie (silnie)
plot_numeric_histogram(titanic_df, 'Parch') #rozkład asymetryczny prawostronnie (silnie)
plot_numeric_histogram(titanic_df, 'Fare') #rozkład asymetryczny prawostronnie (silnie)
plot_numeric_histogram(titanic_df, 'FamilySize') #rozkład asymetryczny prawostronnie (silnie)

